---
Alumna: Castillo Dorantes Sandra Geraldine

## 🏆 Sesion 8 - Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

# Tu codigo aqui:

# Importar librerias
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Definimos la ruta principal
ruta = "C:/Users/HP/Diplomado-ML-Geraldine-Castillo/Modulo_1/datos"

# Carga de archivos
df_cartera = pd.read_csv(ruta + '/cartera_polizas.csv')
df_siniest = pd.read_csv(ruta + '/siniestros.csv')
df_ramos = pd.read_csv(ruta + '/catalogo_ramos.csv')
df_agentes = pd.read_csv(ruta + '/catalogo_agentes.csv')

# Columnas
print(f"========= COLUMNAS TOTALES =========")
print(f"{df_cartera.shape[1]} columnas")
print(df_cartera.columns)
print()

# Revisamos el porcentaje de datos nulos por columnas
print("========= DATOS NULOS POR COLUMNA =========")
print(df_cartera.isna().mean().sort_values(
    ascending = False).reset_index().rename(columns = {'index' : 'columna', 0: 'porcentaje de NA'}).head(10))
print()

# ANALISIS DE NaN
# Tiene sentido que las columnas referentes solo al ramo de autos sean nulas para otras coberturas.
# Debido a esto, se mantienen.
# La columna de deducible es vacía para la cobertura de vida, por eso contiene datos nulos. La 
# mantenemos en nuestros datos.
# La columna prima_neta tiene valores nulos. No se conoce la razon (podría ser error de captura) 
# Se mantiene ya que no tiene gran cantidad de datos nulos y es un dato importante para el analisis.
# La columna ocupación tiene datos nulos, pero son pocos. Se mantiene.
# Las columna nivel_educacion tiene datos nulos. Se desconoce la razon (podría ser error de captura). 
# No es necesaria para nuestro analisis, así que se elimina.
# La columna motivo_baja tiene datos nulos, pero tiene sentido ya que no todas las polizas han sido
# dadas de baja. Los datos se mantienen.

# Observamos que columnas pueden sobrar para nuestro analisis

sobrantes = ['id_contrato_interno',  # Ya tenemos id_poliza y num_poliza para identificar cada registro
             'folio_emision',        # Ya tenemos id_poliza y num_poliza para identificar cada registro
             'id_sistema_legacy',    # Ya tenemos id_poliza y num_poliza para identificar cada registro
             'nombre',               # Ya tenemos la columna nombre_completo
             'apellido_paterno',     # Lo podemos obtener con la columna nombre_completo
             'apellido_materno',     # Lo podemos obtener con la columna nombre_completo
             'nivel_educacion',      # En este momento no es necesario para nuestro analisis actuarial
             'coord_lat',            # No es necesario, a menos que hagamos un analisis por ubicacion
             'coord_long',           # No es necesario, a menos que hagamos un analisis por ubicacion
             'version_documento',    # No es necesario conocer la version del documento
             'hash_documento',       # No es necesario, el id_poliza o num_poliza es suficiente
             'timestamp_carga',      # No es de interes conocer la hora de carga
             'usuario_captura',      # Por el momento nuestro analisis no necesita conocer el usuario de captura
             'ip_carga']             # No necesitamos conocer la ip de la carga

# Columnas necesarias y utiles para nuestro analisis

necesarias = ['id_poliza', 'num_poliza',        # identificadores
              'nombre_completo', 'rfc',         # datos basicos del asegurado
              'fecha_nacimiento', 'edad', 
              'sexo', 'estado_civil', 'ocupacion', 
              'ramo', 'plan', 'fecha_emision',  # datos importantes del seguro
              'fecha_inicio_vigencia', 'fecha_fin_vigencia',
              'num_renovaciones', 'status_poliza', 'canal_venta', 
              'suma_asegurada', 'deducible', 'prima_neta',  # cifras importantes
              'prima_total', 'cuota_prima', 'forma_pago',
              'num_cuotas', 'agente_id', 'estado', 'municipio', 
              'codigo_postal', 'motivo_baja', 'marca_vehiculo', # Autos
              'modelo_vehiculo', 'tipo_vehiculo']

cartera_limpia = pd.read_csv(
    ruta + '/cartera_polizas.csv',
    usecols = necesarias, na_values=['N/D','N/A','ND','--','Sin dato',''])

print(f"========= COLUMNAS UTILES =========")
print(f"{cartera_limpia.columns}")
print()

# Uso de memoria en MB
mb_full = df_cartera.memory_usage(deep=True).sum() / 1024**2
mb_limpio = cartera_limpia.memory_usage(deep=True).sum() / 1024**2

print(f"{'-------------- COMPARACION ----------------':<40}")
print(f"{'DATASET COMPLETO':<20} | |{'DATASET LIMPIO':<20}")
print(f"{'--------------------':<20} | {'--------------------':<20}")
print(f'Filas: {df_cartera.shape[0]:<13} | Filas: {cartera_limpia.shape[0]:<13}')
print(f'Columnas: {df_cartera.shape[1]:<10} | Columnas: {cartera_limpia.shape[1]:<10}')
print(f'Memoria: {mb_full:<8.1f} MB | Memoria: {mb_limpio:<8.1f} MB')
print(f"-------- REDUCCIÓN: {round(mb_full-mb_limpio,2)} MB ({(1-mb_limpio/mb_full)*100:.0f}%) --------")
print()


========= COLUMNAS TOTALES =========
46 columnas
Index(['id_poliza', 'num_poliza', 'id_contrato_interno', 'folio_emision',
       'id_sistema_legacy', 'nombre', 'apellido_paterno', 'apellido_materno',
       'nombre_completo', 'rfc', 'fecha_nacimiento', 'edad', 'sexo',
       'estado_civil', 'ocupacion', 'nivel_educacion', 'ramo', 'plan',
       'fecha_emision', 'fecha_inicio_vigencia', 'fecha_fin_vigencia',
       'num_renovaciones', 'status_poliza', 'motivo_baja', 'canal_venta',
       'marca_vehiculo', 'modelo_vehiculo', 'tipo_vehiculo', 'suma_asegurada',
       'deducible', 'prima_neta', 'prima_total', 'cuota_prima', 'forma_pago',
       'num_cuotas', 'agente_id', 'estado', 'municipio', 'codigo_postal',
       'coord_lat', 'coord_lon', 'version_documento', 'hash_documento',
       'timestamp_carga', 'usuario_captura', 'ip_carga'],
      dtype='object')

========= DATOS NULOS POR COLUMNA =========
             columna  porcentaje de NA
0        motivo_baja           0.95070
1      t

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

# Tu codigo aqui:

# ---- 2a. ELIMINAMOS DUPLICADOS DE CARTERA ----
print(f"DUPLICADOS: {len(cartera_limpia[cartera_limpia.duplicated()])}")
cartera = cartera_limpia.drop_duplicates()
print()

# ---- 2b. NORMALIZAMOS SEXO ----

# Revisamos cuantos valores unicos tenemos en la columna sexo
print("---- NORMALIZAR SEXO ----")
print(f"Antes del mapeo: {cartera['sexo'].unique()}")
cartera['sexo'] = cartera['sexo'].str.strip().str.upper()

# Mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
cartera['sexo'] = cartera['sexo'].map(MAPA_SEXO)

# Observamos valores unicos despues del mapeo
print(f"Despues del mapeo: {cartera['sexo'].unique()}")
print()

# ---- 2c. CONVERTIMOS LAS FECHAS A DATETIME ----
print("------ CONVERTIR FECHAS A DATETIME -----")
print()

# Analizamos el tipo de dato de cada columna
print("Antes de convertir")
col_fechas = ['fecha_nacimiento','fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']
for col_fecha in col_fechas:
    print(f"Columna: {col_fecha} - Tipo: {cartera[col_fecha].dtype}")
print()

# Observamos los formatos de cada columna
print("Columnas con fechas")
print(cartera[col_fechas].head(5))
print()
print("La columna 'fecha_nacimiento' tiene formato distinto, así que la convertimos por separado")
print()

# La columna 'fecha_nacimiento' tiene formato distinto, 
# así que la convertimos por separado
cartera['fecha_nacimiento'] = pd.to_datetime(
    cartera['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce')

# Convertimos las demas columnas con fechas a tipo datetime
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    cartera[col_fecha] = pd.to_datetime(cartera[col_fecha], errors='coerce')

# Observamos el tipo de dato de las columnas despues de convertir
print("Despues de convertir")
for col_fecha in col_fechas:
    print(f"Columna: {col_fecha} - Tipo: {cartera[col_fecha].dtype}")
print()

print("Columnas con fechas")
print(cartera[col_fechas].head(5))
print()

# ---- 2d. RELLENAMOS NaN DE LA COLUMNA PRIMA_NETA ----
print("----- RELLENAR NaN DE PRIMA_NETA -----")
print()
print(f"NaN antes de la limpieza")
print(cartera.groupby('ramo')['prima_neta'].apply(lambda x: x.isna().sum()))
# Limpieza
cartera['prima_neta'] = cartera.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
print()
print(f"NaN despues de la limpieza")
print(cartera.groupby('ramo')['prima_neta'].apply(lambda x: x.isna().sum()))
print()

# ---- 2e. LIMPIAR CODIGO POSTAL (reemplaza 'N/D' con NaN) -----
print("----- LIMPIAR CODIGO POSTAL -----")
print()
print(f"Antes de la limpieza")
print(f"Valores N/D: {len(cartera[cartera['codigo_postal'] == 'N/D'])}")
cartera['codigo_postal'] = cartera['codigo_postal'].replace('N/D', np.nan)
print()
print(f"Despues de la limpieza")
print(f"Valores N/D: {len(cartera[cartera['codigo_postal'] == 'N/D'])}")
print()

# ---- 2f. APLICAMOS OPTIMIZACION DE CATEGORICAS (sin tocar float64 de primas) -----
print("----- OPTIMIZACION DE COLUMNAS CATEGORICAS -----")
print()

cartera_opt = cartera.copy()

categoricas = ['ramo','plan','status_poliza','sexo','canal_venta',
               'forma_pago','estado','estado_civil','tipo_vehiculo']
print(f" {'Columna ':<25}: {'Valores unicos'}")
for col in categoricas:
    if col in cartera_opt.columns:
        n_uniq = cartera_opt[col].nunique()
        n_tot  = len(cartera_opt)
        pct = cartera_opt[col].nunique() / len(cartera_opt)  # # Proporción de valores únicos sobre el total
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct*100}%) → category')
        cartera_opt[col] = cartera_opt[col].astype('category') # Convertimos las columnas en categoricas
print("Los valores unicos por columna son muy pocos. Convertimos las columnas a categoricas")
print()

# Podemos convertir la columna num_renovaciones a entero
cartera_opt['num_renovaciones'] = cartera_opt['num_renovaciones'].fillna(0).astype('int8')
# Dejamos intactas las columnas con varios decimales

# Revisamos la optimizacion
print()
print('Memoria despues de optimizar:')
mb_antes = cartera.memory_usage(deep=True).sum()/1024**2
mb_desp  = cartera_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')
print()

cartera = cartera_opt

DUPLICADOS: 0

---- NORMALIZAR SEXO ----
Antes del mapeo: ['F' 'Femenino' 'M' 'Masculino']
Despues del mapeo: ['F' 'M']

------ CONVERTIR FECHAS A DATETIME -----

Antes de convertir
Columna: fecha_nacimiento - Tipo: object
Columna: fecha_emision - Tipo: object
Columna: fecha_inicio_vigencia - Tipo: object
Columna: fecha_fin_vigencia - Tipo: object

Columnas con fechas
  fecha_nacimiento fecha_emision fecha_inicio_vigencia fecha_fin_vigencia
0       29/04/2002    2021-11-23            2021-11-23         2022-11-23
1       15/08/2002    2019-08-19            2019-08-19         2020-08-19
2       18/10/1994    2022-07-11            2022-07-11         2023-07-11
3       04/02/1986    2019-03-12            2019-03-12         2020-03-12
4       22/07/1996    2020-11-03            2020-11-03         2021-11-03

La columna 'fecha_nacimiento' tiene formato distinto, así que la convertimos por separado

Despues de convertir
Columna: fecha_nacimiento - Tipo: datetime64[ns]
Columna: fecha_emision 

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

# Tu codigo aqui:

# ----- 3a. MERGE CON CATALOGO_RAMOS -----: agregar nombre_largo, tasa_base
cartera = pd.merge(cartera, df_ramos[['ramo', 'nombre_largo', 'tasa_base']], on = 'ramo', how = 'left')

# ----- 3b. MERGE CON CATALOGO_AGENTES -----: agregar nombre del agente, region
cartera = pd.merge(cartera, df_agentes[['agente_id','nombre','region']].rename(
    columns={'nombre':'nombre_agente','region':'region_agente'}),
    on = 'agente_id', how = 'left')

# ----- 3c. CREAR G_EDAD: ----- g_edad con pd.cut
cartera['g_edad'] = pd.cut(cartera['edad'], bins=[0,30,45,60,100], labels=['18-30','31-45','46-60','61+'])

# ----- 3d. CREAR PRIMA_CALC ----- = suma_asegurada * tasa_base * 1.16
cartera['prima_calc'] = cartera['suma_asegurada'] * cartera ['tasa_base'] * 1.16

# ----- 3e. CREAR NIVEL_RIESGO ----- con .apply(clasificar_riesgo) — de mi_modulo
from mi_modulo import clasificar_riesgo
num_siniest = df_siniest.groupby('id_poliza').size().reset_index(name='num_siniest')
cartera = pd.merge(cartera, num_siniest, on = 'id_poliza', how = 'left')
cartera['num_siniest'] = cartera['num_siniest'].fillna(0)
cartera['nivel_riesgo'] = cartera['num_siniest'].apply(clasificar_riesgo)

# ----- 3f. CREAR EDAD_CALC ----- desde fecha_nacimiento
hoy = pd.Timestamp.today()
cartera['edad_calc'] = ((hoy - cartera['fecha_nacimiento']).dt.days / 365.25).round().astype('Int8')

# ----- 3g. CREAR DIAS_VIGENCIA Y FRACCION_EXPUESTA -----
cartera['dias_vigencia'] = (cartera['fecha_fin_vigencia'] - cartera['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - cartera['fecha_inicio_vigencia']).dt.days
cartera['fraccion_expuesta'] = (dias_transcurridos / cartera['dias_vigencia']).clip(0, 1).round(4)

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

# Tu codigo aqui:

# -------- 4a. groupby+agg POR RAMO --------
reporte_ramo = cartera.groupby('ramo').agg(
    polizas       = ('id_poliza',   'count'),
    prima_total   = ('prima_total', 'sum'),
    prima_prom    = ('prima_total', 'mean')
    ).round(2).reset_index()

total = reporte_ramo['prima_total'].sum()
reporte_ramo['pct_cartera'] = reporte_ramo.apply(lambda x: x['prima_total'] / total * 100, axis = 1)

print("--------- REPORTE POR RAMO ---------")
print(reporte_ramo.head(5))
print()

# -------- 4b. groupby+agg POR AGENTE --------
reporte_agente = cartera.groupby('nombre_agente').agg(
    polizas       = ('id_poliza',   'count'),
    prima_total   = ('prima_total', 'sum'),
    prima_prom    = ('prima_total', 'mean')
    ).round(2).reset_index()
reporte_agente['comision'] = (reporte_agente['prima_total'] * 0.10).round(2)

print("--------- REPORTE POR AGENTE ---------")
print(reporte_agente.sort_values(by='polizas', ascending = False).head(5))
print()

# -------- 4c. pivot_table prima por ramo x g_edad --------
tabla_prima = pd.pivot_table(
    cartera,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000
print('------ PRIMA TOTAL POR RAMO Y GRUPO DE EDAD (miles MXN) -----')
print(tabla_prima.to_string())
print()

# -------- 4d. pivot_table polizas por estado x ramo --------
tabla_canal = pd.pivot_table(
    cartera,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('------ POLIZAS POR ESTADO Y CANAL DE VENTA ------')
print(tabla_canal.to_string())
print()

# -------- 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia
max_ramo = reporte_ramo.loc[reporte_ramo['prima_total'].idxmax(), 'ramo']
max_zona = tabla_canal.drop(index='TOTAL')['TOTAL'].idxmax()
print(f"El ramo con mayor prima neta es {max_ramo}")
print(f"El estado con mayor frecuencia es {max_zona}")
print()

--------- REPORTE POR RAMO ---------
                    ramo  polizas   prima_total  prima_prom  pct_cartera
0  Accidentes Personales     5207  2.498068e+07     4797.52     1.706146
1                  Autos    14752  2.538235e+08    17206.04    17.335793
2                    GMM    22531  7.310223e+08    32445.18    49.927809
3                   Vida     7510  4.543321e+08    60496.95    31.030253

--------- REPORTE POR AGENTE ---------
       nombre_agente  polizas  prima_total  prima_prom    comision
74       Sofia Perez     1231  37442769.63    30416.55  3744276.96
32       Jose Medina     1227  34800214.16    28362.03  3480021.42
76    Victor Jimenez     1207  34242602.28    28370.01  3424260.23
46  Manuel Dominguez      684  19852287.17    29023.81  1985228.72
67     Sandra Garcia      679  20269987.60    29852.71  2026998.76

------ PRIMA TOTAL POR RAMO Y GRUPO DE EDAD (miles MXN) -----
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo      

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

# Tu codigo aqui:

import os, time

with pd.ExcelWriter(ruta + '/cartera_q1_2026_final.xlsx', engine='openpyxl') as writer:
    cartera.to_excel(writer, sheet_name='Cartera_Limpia', index=False)
    reporte_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    reporte_agente.to_excel(writer, sheet_name='Resumen_Agente', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Zona')

formatos = {
    'CSV': (ruta + '/cartera_q1_2026_final.csv',
            lambda: cartera.to_csv(ruta + '/cartera_q1_2026_final.csv', index=False)),
    'Parquet': (ruta + '/cartera_q1_2026_final.parquet',
                lambda: cartera.to_parquet(ruta + '/cartera_q1_2026_final.parquet', index=False))}

print(f'{"Formato":<10} {"Tamanio":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (rutas, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(rutas)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')


Formato       Tamanio     Tiempo
-----------------------------------
CSV           16543 KB     1352 ms
Parquet        3395 KB      510 ms
